In [2]:
import geogridfusion
import pvdeg

import numpy as np # range generation for bounding box

conn = geogridfusion.start()

Starting Postgres subprocess...
PostgreSQL connection established after 0.10 seconds.
postgis already installed


Generate coordinate pairs to request

In [ ]:
longitude = [-109.060253, -102.041524]
latitude = [36.992426, 41.003444]

RESOLUTION = 5

lats = np.linspace(latitude[0], latitude[1], RESOLUTION)
lons = np.linspace(longitude[0], longitude[1], RESOLUTION)

lat_grid, lon_grid = np.meshgrid(lats, lons, indexing='ij')
pairs = np.column_stack((lat_grid.ravel(), lon_grid.ravel()))


option a: get weather and store for one location at a time.

In [ ]:
for pair in pairs:
    single_weather, single_meta = pvdeg.weather.get(
        database="PVGIS",
        id=tuple(pair)
    )

    geogridfusion.store_single(conn=conn, weather_df=single_weather, meta=single_meta, tmy=True, source_name="pvgis")


option b: get weather using dask for parallel speedup and write individually after all locations are loaded

In [ ]:
client = pvdeg.geospatial.start_dask()

# this is a required step, otherwis4e the distrubted weather call will raise AttributeError: 'NoneType' object has no attribute 'sizes'
pairs_tuples = [tuple(pair) for pair in pairs]

try:
    geo_weather, geo_meta, failed_idx = pvdeg.weather.weather_distributed(
        database="PVGIS",
        coords=pairs_tuples
    )

except Exception as e:
    client.close()
    raise e


# geo_weather

for i, gid in enumerate(geo_weather.gid):
    geogridfusion.store_single(
        conn=conn,
        weather_df=geo_weather.sel(gid=gid).to_dataframe(),
        meta=geo_meta.iloc[i].to_dict(),
        tmy=True,
        source_name='pvgis'
    )

c:\Users\tford\AppData\Local\miniconda3\envs\geogridfusion\Lib\site-packages\distributed\node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 50853 instead
  warnings.warn(
c:\Users\tford\AppData\Local\miniconda3\envs\geogridfusion\Lib\contextlib.py:144: UserWarning: Creating scratch directories is taking a surprisingly long time. (1.68s) This is often due to running workers on a network file system. Consider specifying a local-directory to point workers to write scratch data to a local disk.
  next(self.gen)


Dashboard: http://127.0.0.1:50853/status
Connected to a Dask scheduler | Dashboard: http://127.0.0.1:50853/status


Load all stored locations from dataset

In [ ]:
loaded_weather, loaded_meta = geogridfusion.load_many(conn=conn, source_name="pvgis")

loaded_weather

Run pvdeg geospatial degradation analysis

In [ ]:
pvdeg.geospatial.analysis(...)

Plot result of small analysis

In [ ]:
plt.plot(...)

### But wait, what if we want to do the entire country

In [ ]:
# download 100 points for the rest of the country
# should they be from pvgis or nsrdb, or others?
...

# store 100 points for the rest of the country

Load whole country, (show how we can store more points over-time as needs change)

Can demonstrate some downselection here? Or combining of different datasets?

In [ ]:
geogridfusion.load_many(conn=conn, source_name="pvgis")

Perform same analysis and create a plot for the whole country.

In [ ]:
pvdeg.geospatial.analysis()

plt.plot(...)